# 08 - Demo (hero figure and a live run for the demo video)
Runtime: **CPU** is enough. Run the cells one at a time.
1. Builds the README hero figure from the cached predictions.
2. Runs the baseline and the degradation-aware model live on a few test chips,
   clean and with heavy speckle, side by side.
3. Shows the robustness comparison plot.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
from google.colab import userdata

try:
    token = userdata.get('GH_TOKEN')  # only needed while the repo is private
except Exception:
    token = None
AUTH = f'{token}@' if token else ''
REPO = 'ThemistoklisSakkas/sar-ship-robustness'
DEST = '/content/sar-ship-robustness'
if os.path.isdir(DEST):
    !git -C {DEST} pull -q https://{AUTH}github.com/{REPO}.git main
else:
    !git clone -q https://{AUTH}github.com/{REPO}.git {DEST}
    !git -C {DEST} remote set-url origin https://github.com/{REPO}.git
%cd {DEST}
!pip install -q -r requirements.txt

DRIVE_ROOT = '/content/drive/MyDrive/sar-ship-robustness'
DATA_ROOT = f'{DRIVE_ROOT}/data'
RUNS_ROOT = f'{DRIVE_ROOT}/runs'

## 1. Hero figure for the README
Saved to `report_figures/00_hero.png` on Drive: download it into `docs/figures/` of the repo.

In [ ]:
!python -m sarship.hero --data-root {DATA_ROOT} --runs-root {RUNS_ROOT} --out {DRIVE_ROOT}/report_figures/00_hero.png
from IPython.display import Image, display
display(Image(f'{DRIVE_ROOT}/report_figures/00_hero.png'))

## 2. Live run: clean vs heavy speckle (L = 1), baseline vs degradation-aware model

In [ ]:
import cv2, numpy as np, matplotlib.pyplot as plt
from ultralytics import YOLO
from sarship.degrade import speckle

models = {'baseline': YOLO(f'{RUNS_ROOT}/baseline_yolo11n/weights/best.pt'),
          'degradation-aware': YOLO(f'{RUNS_ROOT}/robust_yolo11n/weights/best.pt')}
test_dir = f'{DATA_ROOT}/ssdd_yolo/images/test'
chips = ['000739', '001111', '000219']  # offshore group, offshore spread, dense port

rng = np.random.default_rng(42)
fig, axes = plt.subplots(len(chips), 4, figsize=(20, 5 * len(chips)))
for row, stem in enumerate(chips):
    path = next(p for p in os.listdir(test_dir) if p.startswith(stem))
    clean = cv2.imread(f'{test_dir}/{path}', cv2.IMREAD_GRAYSCALE)
    noisy = speckle(clean, looks=1, rng=rng)
    col = 0
    for name, model in models.items():
        for label, img in (('clean', clean), ('speckle L=1', noisy)):
            r = model.predict(cv2.cvtColor(img, cv2.COLOR_GRAY2BGR), conf=0.25, verbose=False)[0]
            axes[row, col].imshow(r.plot(labels=False)[..., ::-1])
            axes[row, col].set_title(f'{name} | {label} | {len(r.boxes)} detections')
            axes[row, col].axis('off')
            col += 1
plt.tight_layout()
plt.show()

## 3. The robustness result (25 degraded test conditions)

In [ ]:
display(Image(f'{RUNS_ROOT}/robust_yolo11n/comparison/comparison.png'))